In [1]:
TICKER = "AAPL"
BENCHMARK = "^GSPC"
START = "2015-01-01"

In [2]:
import yfinance as yf
import pandas as pd

raw = yf.download([TICKER, BENCHMARK], start=START, auto_adjust=True, progress=False)
close = raw["Close"]
close.head()

Ticker,AAPL,^GSPC
Date,,
2015-01-02,24.171762,2058.199951
2015-01-05,23.490797,2020.579956
2015-01-06,23.493010,2002.609985
2015-01-07,23.822432,2025.900024
2015-01-08,24.737738,2062.139893


In [3]:
prices = (
    close.reset_index()
         .melt(id_vars="Date", var_name="ticker", value_name="close")
         .rename(columns={"Date": "date"})
         .dropna()
)
prices["date"] = prices["date"].dt.strftime("%Y-%m-%d")
prices.head()

,date,ticker,close
0,2015-01-02,AAPL,24.171762
1,2015-01-05,AAPL,23.490797
2,2015-01-06,AAPL,23.493010
3,2015-01-07,AAPL,23.822432
4,2015-01-08,AAPL,24.737738


In [4]:
import sqlite3
from pathlib import Path

DB_PATH = Path.cwd().parent / "data" / "prices.db"

with sqlite3.connect(DB_PATH) as conn:
    prices.to_sql("prices", conn, if_exists="replace", index=False)

In [8]:
with sqlite3.connect(DB_PATH) as conn:
    check = pd.read_sql("SELECT ticker, COUNT(*) AS n FROM prices GROUP BY ticker", conn)
check

,ticker,n
0,AAPL,2935
1,^GSPC,2935
